# Setup

In [1]:
import json

from torch.cuda import is_available as is_cuda_available
import numpy as np
from lighthouse.models import CGDETRPredictor, QDDETRPredictor
from utils.helper_methods import split_video_ffmpeg, parse_timestamps, extract_clips, adjust_video_speed
import pandas as pd
import os

C:\Users\denve\miniconda3\envs\moment-retrieval-and-video-summarization\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model weights can be downloaded from https://zenodo.org/records/13960580
Checkpoint file should correspond to the feature selected in predictor

For slowfast video encoding features additional weights should be downloaded. They can be found in lighthouse GitHub page (https://github.com/line/lighthouse/tree/main) or downloaded from [here](https://dl.fbaipublicfiles.com/pyslowfast/model_zoo/kinetics400/SLOWFAST_8x8_R50.pkl)

### Note: If you don't have a GPU available, do not use clip+slowfast, as it will take a really long time to run

In [2]:
device = "cuda" if is_cuda_available() else "cpu"
device

'cuda'

In [3]:
# model = QDDETRPredictor('model_weights/clip_qd_detr_qvhighlight.ckpt', device=device,
#                         feature_name='clip')

model = CGDETRPredictor('model_weights/clip_slowfast_cg_detr_qvhighlight.ckpt', device=device,
                        feature_name='clip_slowfast', slowfast_path="model_weights/SLOWFAST_8x8_R50.pkl", pann_path=None)

Loading the queries. You can use commented code to run the pipeline on human annotations(comment out the currently active cell for that)

In [5]:
# set the original video path
video_name = "video_26"
og_video_path = f"TVSum/{video_name}.mp4"

In [4]:
# Load annotations for testing
# events = pd.read_csv("Annotations/7-TVSum-video_25-i6324060.txt")[['annotation', 'timestamp']]
# events['timestamp'].replace(np.nan, None)
# events['timestamp'] = events['timestamp'].map(parse_timestamps)
# events['annotation'] = events['annotation'].map(lambda x: [x])

In [6]:
# Use results from VLM/LLM pipeline

with open(f"Descriptions/{video_name}.json", 'r', encoding='utf-8') as f:
    events_json = json.load(f)

events_full = pd.json_normalize(events_json, record_path=['llm_results'])[['prompt_type', 'vlm_source', 'vlm_frame_count', 'parsed_events', 'parsed_events_clean']]

events_full = events_full[(events_full['prompt_type'] == 'concise') & (events_full['vlm_frame_count'] == 50) & (events_full['vlm_source'] == 'long')] # we only select the most optimized parameters from the vlm/llm pipeline
if all(events_full['parsed_events_clean'].notna()):
    events = events_full['parsed_events_clean'].to_frame()
else:
    events = events_full['parsed_events'].to_frame()
events['timestamps'] = np.nan

events

,parsed_events,timestamps
16,[A young girl gets off an elevator onto a subw...,NaN


# Video preparation
CG-DETR model has a major limitation -- it doesn't allow videos longer than 150 seconds. We will try several approaches to this problem

## Splitting
The most obvious way is splitting videos in chunks. We cut the original video in segments no longer than 150 seconds and then process them seperately

In [7]:
output_path = f"processed_videos/{video_name}_chunks"
split_video_ffmpeg(og_video_path, output_path, 150)

Splitting TVSum/video_26.mp4 into 150s chunks...
Chunks are successfully saved into 'processed_videos/video_26_chunks' folder.


'processed_videos/video_26_chunks'

In [8]:
results = []
for row in events.iterrows():
    results_per_entry = [] # results for every event list

    queries, true_timestamps = row[1]
    for query in queries:
        results_per_chunk = [] # results for every query

        print(f"Query: {query}")
        print(f"True timestamp: {true_timestamps}")

        for index, chunk in enumerate(os.listdir(output_path)):
            if chunk.endswith('.mp4'):
                video_path = output_path + '/' + chunk
                video = model.encode_video(video_path)
                prediction = model.predict(query, video)
                results_per_chunk.append((index, prediction))

                print("\tChunk:", index)
                print(f"\t\tClosest Moment: {prediction['pred_relevant_windows'][0][0]}, {prediction['pred_relevant_windows'][0][1]}, Corresponding Confidence score: {prediction['pred_relevant_windows'][0][2]}")

        print('-' * 50)

        results_per_entry.append(results_per_chunk)

    results.append(results_per_entry)

Query: A young girl gets off an elevator onto a subway platform, observing her surroundings.
True timestamp: nan
	Chunk: 0
		Closest Moment: 18.9366, 27.1572, Corresponding Confidence score: 0.9992
--------------------------------------------------
Query: She finds another person wearing a beige coat standing next to someone who has fallen asleep on the subway platform.
True timestamp: nan
	Chunk: 0
		Closest Moment: 58.6896, 74.6353, Corresponding Confidence score: 0.961
--------------------------------------------------
Query: The scene shifts indoors again, showing two men playing bagpipes outside near buildings.
True timestamp: nan
	Chunk: 0
		Closest Moment: 13.4727, 26.7983, Corresponding Confidence score: 0.9984
--------------------------------------------------
Query: Viewers see train windows looking towards tall apartment blocks under overcast skies as trains pass by in the background.
True timestamp: nan
	Chunk: 0
		Closest Moment: 0.1322, 6.5648, Corresponding Confidence sc

Now we have a problem, because we get a timestamp for each chunk, but we only need one timestamp for the whole video. The simplest solution is to pick confidence score, although this approach has a lot of shortcomings, we will use it, as alternatives require more complex algorithms

Keep in mind that extracting timestamps for a lot of queries may take quite a while

In [9]:
chunks_timestamp_results = []
for results_per_entry in results:

    max_results_per_entry = []
    for results_per_chunk in results_per_entry:
        max_score = 0.0
        best_timestamp = ()
        for chunk in results_per_chunk:
            prediction = chunk[1]
            if prediction['pred_relevant_windows'][0][2] > max_score:
                max_score = prediction['pred_relevant_windows'][0][2]
                best_timestamp = (prediction['pred_relevant_windows'][0][0] + 150 * chunk[0], prediction['pred_relevant_windows'][0][1] + 150 * chunk[0]) # we add 150 multiplied by chunk index to get the global timestamp(the model returns timestamp local to a particular chunk)
        max_results_per_entry.append(best_timestamp)

    chunks_timestamp_results.append(max_results_per_entry)

chunks_timestamp_results

[[(18.9366, 27.1572),
  (58.6896, 74.6353),
  (13.4727, 26.7983),
  (0.1322, 6.5648),
  (96.1163, 102.5111),
  (14.0228, 24.621),
  (28.5261, 50.8526),
  (27.8549, 48.8653),
  (17.767, 25.7743)]]

## Speeding up
Another approach is speeding up the video until it becomes short enough

In [10]:
output_path = f"processed_videos/{video_name}_sped_up.mp4"
speed_factor = adjust_video_speed(og_video_path, output_path, 150)

Video is 110.53s. No adjustment needed.


In [11]:
sped_up_timestamps_results = []

for row in events.iterrows():
    queries, timestamps = row[1]
    predicted_timestamps = []


    results_per_entry = []
    for query in queries:
        print(f"Query: {query}")
        print(f"True timestamp: {query}")

        video = model.encode_video(output_path)
        prediction = model.predict(query, video)

        print(f"\t\tClosest Moment: {prediction['pred_relevant_windows'][0][0], prediction['pred_relevant_windows'][0][1]}, Corresponding Confidence score: {prediction['pred_relevant_windows'][0][2]}")

        results_per_entry.append((prediction['pred_relevant_windows'][0][0] * speed_factor, prediction['pred_relevant_windows'][0][1] * speed_factor))
        print("-" * 50)

    sped_up_timestamps_results.append(results_per_entry)

Query: A young girl gets off an elevator onto a subway platform, observing her surroundings.
True timestamp: A young girl gets off an elevator onto a subway platform, observing her surroundings.
		Closest Moment: (18.8242, 26.5767), Corresponding Confidence score: 0.9992
--------------------------------------------------
Query: She finds another person wearing a beige coat standing next to someone who has fallen asleep on the subway platform.
True timestamp: She finds another person wearing a beige coat standing next to someone who has fallen asleep on the subway platform.
		Closest Moment: (58.7302, 77.1017), Corresponding Confidence score: 0.9354
--------------------------------------------------
Query: The scene shifts indoors again, showing two men playing bagpipes outside near buildings.
True timestamp: The scene shifts indoors again, showing two men playing bagpipes outside near buildings.
		Closest Moment: (15.5114, 25.1292), Corresponding Confidence score: 0.9876
--------------

In [12]:
events_full['predicted_timestamps_chunks'] = chunks_timestamp_results
events_full['predicted_timestamps_sped_up'] = sped_up_timestamps_results

In [13]:
events_full.to_json(f"Timestamps/{og_video_path.split('/')[-1].split('.')[0]}_with_predictions.json", orient='records')

# Video Summary
This will automatically create a video summary using the timestamps we extracted

In [14]:
for row in events_full.iterrows():
    chunks_timestamps = row[1]['predicted_timestamps_chunks']
    extract_clips(og_video_path, f"{video_name}_chunks_summary_{row[0]}.mp4", chunks_timestamps)

    sped_up_timestamps = row[1]['predicted_timestamps_sped_up']
    extract_clips(og_video_path, f"{video_name}_sped_up_summary_{row[0]}.mp4", sped_up_timestamps)

Processing clips...
Video successfully saved as video_26_chunks_summary.mp4
Processing clips...
Video successfully saved as video_26_sped_up_summary.mp4
